In [17]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [18]:
df = pd.read_csv('honeywell_gold_dataset.csv')
df.shape
df.head()


,timestamp,q[0],q[1],q[2],q[3],delta_q_reset[0],delta_q_reset[1],delta_q_reset[2],delta_q_reset[3],quat_reset_counter,...,xy_reset_counter,z_reset_counter,vxy_reset_counter,vz_reset_counter,heading_reset_counter,xy_global,z_global,dist_bottom_valid,label,scenario_type
0,0,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,2,0,2,1,3,1,1,1.0,0,clean_flight
1,1,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,2,0,2,1,3,1,1,1.0,0,clean_flight
2,2,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,2,0,2,1,3,1,1,1.0,0,clean_flight
3,3,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,2,0,2,1,3,1,1,1.0,0,clean_flight
4,4,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,2,0,2,1,3,1,1,1.0,0,clean_flight


In [19]:
from sklearn.preprocessing import LabelEncoder
import sklearn as sk

X = df.drop(columns=[
    'label', 'timestamp', 'scenario_type',
    'lat_x', 'lon_x', 'alt_x',
    'lat_y', 'lon_y', 'alt_y'
])
y = df['label']


le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

In [20]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols)
    ]
)
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [21]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    'DT' : DecisionTreeClassifier(random_state=42),
    'RF' : RandomForestClassifier(random_state=42),
    'XGBoost' : XGBClassifier(eval_metric='mlogloss',random_state=42),
    'CatBoost' : CatBoostClassifier(verbose=0,random_seed=42),
    'LightGBM' : LGBMClassifier(verbose=-1,random_seed=42),
}

In [22]:
from sklearn.metrics import accuracy_score, f1_score

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'{name}: Accuracy: {str(accuracy_score(y_test, y_pred))}, F1: {str(f1_score(y_test, y_pred))}')

DT: Accuracy: 0.7009401880376075, F1: 0.7126657697482222
RF: Accuracy: 0.7101420284056812, F1: 0.735632183908046
XGBoost: Accuracy: 0.7501500300060012, F1: 0.775883725103176
CatBoost: Accuracy: 0.7815563112622524, F1: 0.8073394495412844
LightGBM: Accuracy: 0.7889577915583117, F1: 0.8150096440469928


In [23]:
proba_cat = models['CatBoost'].predict_proba(X_test)[:, 1]
proba_lgb = models['LightGBM'].predict_proba(X_test)[:, 1]
proba_xgb = models['XGBoost'].predict_proba(X_test)[:, 1]

avg_proba = (proba_cat + proba_lgb + proba_xgb) / 3.0

y_pred_ensemble = (avg_proba >= 0.5).astype(int)

print("--------------------------------------------------")
print("Ensemble (Soft Voting) - Accuracy: " + str(accuracy_score(y_test, y_pred_ensemble)))
print("Ensemble (Soft Voting) - F1: " + str(f1_score(y_test, y_pred_ensemble)))

--------------------------------------------------
Ensemble (Soft Voting) - Accuracy: 0.7811562312462492
Ensemble (Soft Voting) - F1: 0.8072586328400282


In [24]:
pred_cat = models['CatBoost'].predict(X_test)
pred_lgb = models['LightGBM'].predict(X_test)
pred_xgb = models['XGBoost'].predict(X_test)

total_votes = pred_cat + pred_lgb + pred_xgb

y_pred_hard_voting = (total_votes >= 2).astype(int)

print("--------------------------------------------------")
print("Ensemble (Hard Voting) - Accuracy: " + str(accuracy_score(y_test, y_pred_hard_voting)))
print("Ensemble (Hard Voting) - F1: " + str(f1_score(y_test, y_pred_hard_voting)))

--------------------------------------------------
Ensemble (Hard Voting) - Accuracy: 0.7841568313662732
Ensemble (Hard Voting) - F1: 0.8098678414096916


In [25]:
xgb_model = models['XGBoost']
importances = pd.Series(xgb_model.feature_importances_, index=preprocessor.get_feature_names_out())

print(importances.sort_values(ascending=False).head(10))

num__alt_ellipsoid_x    0.447108
num__c_variance_rad     0.124853
num__vel_n_m_s          0.100986
num__dist_bottom        0.083466
num__alt_ellipsoid_y    0.027545
num__eph_x              0.011669
num__epv_x              0.010883
num__y                  0.008367
num__q[2]               0.007684
num__vy                 0.007589
dtype: float32
